# Deepfake Detection Analysis

A comparative study of CNN and ResNet-50 architectures for Deepfake detection.  
This notebook walks through:
1. Data loading and exploration
2. Preprocessing (histogram equalization + sharpening)
3. Model training (CNN baseline & ResNet-50)
4. Evaluation and confusion-matrix visualisation

## 1. Imports

In [ ]:
import os
import sys

import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Add the project root to the path so we can import from /src
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocessing import apply_histogram_equalization, apply_sharpening

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)

## 2. Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR     = os.path.join(PROJECT_ROOT, "data")       # raw dataset root
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "results")    # output artefacts
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Hyper-parameters ─────────────────────────────────────────────────────────
IMG_SIZE   = (128, 128)   # (height, width) fed to the models
BATCH_SIZE = 32
EPOCHS     = 20
SEED       = 42

CLASSES    = ["real", "fake"]

## 3. Preprocessing Utilities Demo

Demonstrate the preprocessing functions from `src/preprocessing.py` on a sample image.

In [ ]:
def demo_preprocessing(image_path: str) -> None:
    """Display the original image alongside preprocessed variants."""
    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not load image at '{image_path}'. Skipping demo.")
        return

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Preprocessing variants
    hist_clahe   = cv2.cvtColor(apply_histogram_equalization(img, method="clahe"),
                                cv2.COLOR_BGR2RGB)
    hist_global  = cv2.cvtColor(apply_histogram_equalization(img, method="global"),
                                cv2.COLOR_BGR2RGB)
    sharp_std    = cv2.cvtColor(apply_sharpening(img, kernel_name="standard"),
                                cv2.COLOR_BGR2RGB)
    sharp_strong = cv2.cvtColor(apply_sharpening(img, kernel_name="strong"),
                                cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    titles  = ["Original", "CLAHE", "Global EQ", "Sharpen (std)", "Sharpen (strong)"]
    images  = [img_rgb, hist_clahe, hist_global, sharp_std, sharp_strong]

    for ax, title, image in zip(axes, titles, images):
        ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")

    plt.suptitle("Preprocessing Comparison", fontsize=14)
    plt.tight_layout()
    plt.show()


# Replace the path below with a real image from your dataset
# demo_preprocessing(os.path.join(DATA_DIR, "sample.jpg"))

## 4. Dataset Loading

Load the dataset from `DATA_DIR` using `tf.keras.utils.image_dataset_from_directory`.  
Expected directory layout:
```
data/
  real/   <- real face images
  fake/   <- AI-generated (deepfake) images
```

In [ ]:
def load_dataset(data_dir, img_size, batch_size, seed, validation_split=0.2):
    """Load image dataset split into training and validation sets."""
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        labels="inferred",
        label_mode="binary",
        class_names=CLASSES,
        color_mode="rgb",
        batch_size=batch_size,
        image_size=img_size,
        shuffle=True,
        seed=seed,
        validation_split=validation_split,
        subset="training",
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        labels="inferred",
        label_mode="binary",
        class_names=CLASSES,
        color_mode="rgb",
        batch_size=batch_size,
        image_size=img_size,
        shuffle=False,
        seed=seed,
        validation_split=validation_split,
        subset="validation",
    )
    # Normalise pixel values to [0, 1]
    normalise = layers.Rescaling(1.0 / 255)
    train_ds  = train_ds.map(lambda x, y: (normalise(x), y),
                             num_parallel_calls=tf.data.AUTOTUNE)
    val_ds    = val_ds.map(lambda x, y: (normalise(x), y),
                           num_parallel_calls=tf.data.AUTOTUNE)
    return (train_ds.cache().prefetch(tf.data.AUTOTUNE),
            val_ds.cache().prefetch(tf.data.AUTOTUNE))


# Uncomment once your data directory is populated:
# train_ds, val_ds = load_dataset(DATA_DIR, IMG_SIZE, BATCH_SIZE, SEED)
# print(f"Training batches : {len(train_ds)}")
# print(f"Validation batches: {len(val_ds)}")

## 5. Model Definitions

### 5a. Custom CNN Baseline

In [ ]:
def build_cnn(input_shape=(128, 128, 3)) -> keras.Model:
    """Lightweight CNN baseline for binary deepfake classification."""
    inputs = keras.Input(shape=input_shape, name="input")

    x = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="output")(x)

    model = keras.Model(inputs, outputs, name="cnn_baseline")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


cnn_model = build_cnn(input_shape=(*IMG_SIZE, 3))
cnn_model.summary()

### 5b. ResNet-50 Transfer Learning

In [ ]:
def build_resnet50(input_shape=(128, 128, 3)) -> keras.Model:
    """ResNet-50 with ImageNet weights for transfer learning."""
    base = keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape,
    )
    # Freeze the convolutional base
    base.trainable = False

    inputs  = keras.Input(shape=input_shape, name="input")
    # ResNet-50 expects inputs scaled to [0, 255]; preprocess accordingly
    x       = keras.applications.resnet50.preprocess_input(inputs * 255.0)
    x       = base(x, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(256, activation="relu")(x)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="output")(x)

    model = keras.Model(inputs, outputs, name="resnet50_transfer")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


resnet_model = build_resnet50(input_shape=(*IMG_SIZE, 3))
resnet_model.summary()

## 6. Training

Uncomment the cells below once the dataset is available.

In [ ]:
# callbacks = [
#     keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
#     keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
# ]
#
# cnn_history = cnn_model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=EPOCHS,
#     callbacks=callbacks,
# )
#
# resnet_history = resnet_model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=EPOCHS,
#     callbacks=callbacks,
# )

## 7. Evaluation & Confusion Matrices

In [ ]:
def plot_and_save_confusion_matrix(model, dataset, model_name: str,
                                   save_dir: str = RESULTS_DIR) -> None:
    """Compute, display, and save a confusion matrix for *model* on *dataset*."""
    y_true, y_pred = [], []
    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_pred.extend((preds.squeeze() > 0.5).astype(int).tolist())
        y_true.extend(labels.numpy().astype(int).tolist())

    cm   = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=CLASSES)

    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f"Confusion Matrix – {model_name}")
    plt.tight_layout()

    save_path = os.path.join(save_dir, f"confusion_matrix_{model_name}.png")
    plt.savefig(save_path, dpi=150)
    print(f"Saved confusion matrix to '{save_path}'")
    plt.show()


# Uncomment once trained:
# plot_and_save_confusion_matrix(cnn_model,    val_ds, "CNN_Baseline")
# plot_and_save_confusion_matrix(resnet_model, val_ds, "ResNet50")

## 8. Training History Visualisation

In [ ]:
def plot_history(history, model_name: str) -> None:
    """Plot accuracy and loss curves for a training *history* object."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Accuracy
    axes[0].plot(history.history["accuracy"],     label="train")
    axes[0].plot(history.history["val_accuracy"], label="val")
    axes[0].set_title(f"{model_name} – Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    # Loss
    axes[1].plot(history.history["loss"],     label="train")
    axes[1].plot(history.history["val_loss"], label="val")
    axes[1].set_title(f"{model_name} – Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


# Uncomment once trained:
# plot_history(cnn_history,    "CNN Baseline")
# plot_history(resnet_history, "ResNet-50")